# ЛР-03: Социальная защита в зимний период

## Student notebook: civil 02

Этот notebook предназначен для самостоятельного анализа.

Готовых численных ответов и заполненного sensitivity-разбора здесь нет.

## 1. Зачем нужен этот кейс

Здесь важно понять, какой ресурс становится главным узким местом зимой.

После выполнения работы студент должен уметь:

1. отделять активные ограничения от тех, где остаётся запас;
2. интерпретировать тени цен без длинного ручного симплекс-анализа;
3. сравнивать прогноз по `shadow price` с фактом;
4. делать управленческий вывод по дефицитности ресурсов.

## 2. Исходные данные

### Ограничения ресурсов

| Ресурс | Лимит |
| --- | --- |
| Бюджет | 96 |
| Трудозатраты | 60 |
| Операционная ёмкость | 50 |

### Программы

| Программа | Эффект | Бюджет | Трудозатраты | Операционная ёмкость |
| --- | --- | --- | --- | --- |
| Пункты обогрева | 97 | 46 | 24 | 18 |
| Продуктовые сертификаты | 76 | 24 | 16 | 9 |
| Социальные патрули | 70 | 18 | 14 | 12 |
| Срочный ремонт жилья | 92 | 44 | 28 | 22 |

## 3. Как читать прямую и двойственную задачи

Прямая задача выбирает масштабы программ `x_j`: это обычный план действий. Двойственная задача смотрит на ту же модель как в зеркале: вместо выбора программ она назначает внутренние цены ограничениям.

- `y_i` относится к ресурсному ограничению и показывает shadow price ресурса.
- `z_j` относится к верхней границе `x_j <= 1` и показывает ценность разрешения расширить программу выше 100 процентов.
- `slack` показывает остаток ресурса, а `binding` показывает, что ресурс исчерпан.

Единица измерения теневой цены: `единицы эффекта / единица соответствующего ограничения`. Нулевая теневая цена означает только то, что дополнительная единица этого ресурса локально не улучшает текущий оптимум. Для больших изменений прогноз нужно проверять повторным решением, потому что shadow price является локальной оценкой.

## 4. Что нужно сделать

1. Запишите прямую модель через переменные масштабов программ.
2. Объясните, почему двойственная задача является зеркальным взглядом на ограничения прямой задачи.
3. Подготовьте `c`, `A_ub`, `b_ub`, `bounds` для `linprog`.
4. Определите активные ограничения и запас ресурса.
5. Кратко запишите двойственную модель с переменными `y_i` и `z_j`.
6. Укажите единицу измерения каждой теневой цены.
7. Проведите минимум два сценария по `b` и один сценарий по `c`.
8. Сравните локальный прогноз по теневой цене с фактическим пересчётом.

In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import linprog

# Шаг 1. Задаем исходные данные прямой задачи.

effects = np.array(
    [
        97,
        76,
        70,
        92,
    ],
    dtype=float,
)

A_ub = np.array(
    [
        [46, 24, 18, 44],
        [24, 16, 14, 28],
        [18, 9, 12, 22],
    ],
    dtype=float,
)

b_ub = np.array(
    [
        96,
        60,
        50,
    ],
    dtype=float,
)

bounds = [(0, 1)] * len(effects)

# Шаг 2. Формируем нейтральные подписи для табличной проверки данных.
program_names = [f"Программа {idx + 1}" for idx in range(len(effects))]
resource_names = [f"Ресурс {idx + 1}" for idx in range(len(b_ub))]

effects_df = pd.DataFrame(
    {
        "программа": program_names,
        "эффект на единицу": effects,
    }
)
A_ub_df = pd.DataFrame(
    A_ub,
    index=resource_names,
    columns=program_names,
)
b_ub_df = pd.DataFrame(
    {
        "ресурс": resource_names,
        "лимит": b_ub,
    }
)

print("Число программ =", len(effects))
print("Число ресурсных ограничений =", len(b_ub))
print("Вектор эффектов:")
display(effects_df)

print("Матрица ресурсных коэффициентов A_ub:")
display(A_ub_df)

print("Вектор правых частей b_ub:")
display(b_ub_df)

Число программ = 4
Число ресурсных ограничений = 3
Вектор эффектов:


,программа,эффект на единицу
0,Программа 1,97.0
1,Программа 2,76.0
2,Программа 3,70.0
3,Программа 4,92.0


Матрица ресурсных коэффициентов A_ub:


,Программа 1,Программа 2,Программа 3,Программа 4
Ресурс 1,46.0,24.0,18.0,44.0
Ресурс 2,24.0,16.0,14.0,28.0
Ресурс 3,18.0,9.0,12.0,22.0


Вектор правых частей b_ub:


,ресурс,лимит
0,Ресурс 1,96.0
1,Ресурс 2,60.0
2,Ресурс 3,50.0


## 5. Шаблон для самостоятельной сборки решения

Сначала решите прямую задачу, потом переходите к binding/slack, dual и sensitivity. На каждом шаге держите в голове зеркало: `x_j` описывает действия, а `y_i` и `z_j` объясняют ценность ограничений.

In [2]:
def solve_primal(effects, A_ub, b_ub, bounds):
    """Решает прямую задачу максимизации через `linprog`."""

    # Переходим к задаче минимизации: min -c^T x
    c = -effects

    result = linprog(
        c,
        A_ub=A_ub,
        b_ub=b_ub,
        bounds=bounds,
        method='highs'
    )

    if not result.success:
        raise RuntimeError(f"Solver failed: {result.message}")

    # Теневые цены (shadow prices) для ограничений ≤
    # Для задачи минимизации: shadow_prices = -result.ineqlin.marginals
    shadow_prices = -result.ineqlin.marginals

    return result, shadow_prices

# Решаем прямую задачу
try:
    result, shadow_prices = solve_primal(effects, A_ub, b_ub, bounds)

    print("=== РЕШЕНИЕ ПРЯМОЙ ЗАДАЧИ ===\n")
    print(f"Статус: {result.message}")
    print(f"Оптимальное значение (максимум эффекта): {-result.fun:.2f}")

    print("\nОптимальный план по программам:")
    for i, name in enumerate(program_names):
        print(f"  {name}: {result.x[i]:.3f} ({result.x[i]*100:.1f}%)")

except Exception as e:
    print(f"Ошибка: {e}")
    result = None
    shadow_prices = None

=== РЕШЕНИЕ ПРЯМОЙ ЗАДАЧИ ===

Статус: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Оптимальное значение (максимум эффекта): 259.73

Оптимальный план по программам:
  Программа 1: 1.000 (100.0%)
  Программа 2: 1.000 (100.0%)
  Программа 3: 1.000 (100.0%)
  Программа 4: 0.182 (18.2%)


In [3]:
def analyze_constraints(result, A_ub, b_ub, resource_names):
    """Анализирует активные ограничения и запасы ресурсов."""

    slack = result.slack
    binding = np.isclose(slack, 0.0, atol=1e-6)

    print("\n=== АНАЛИЗ ОГРАНИЧЕНИЙ ===\n")
    print("Ресурс | Использовано | Лимит | Остаток | Статус")
    print("-" * 60)

    for i, name in enumerate(resource_names):
        used = (A_ub[i] * result.x).sum()
        status = "АКТИВНО (binding)" if binding[i] else f"Свободно ({slack[i]:.1f} ед.)"
        print(f"{name:12} | {used:11.2f} | {b_ub[i]:5.0f} | {slack[i]:7.2f} | {status}")

    return slack, binding

if result is not None:
    slack, binding = analyze_constraints(result, A_ub, b_ub, resource_names)


=== АНАЛИЗ ОГРАНИЧЕНИЙ ===

Ресурс | Использовано | Лимит | Остаток | Статус
------------------------------------------------------------
Ресурс 1     |       96.00 |    96 |    0.00 | АКТИВНО (binding)
Ресурс 2     |       59.09 |    60 |    0.91 | Свободно (0.9 ед.)
Ресурс 3     |       43.00 |    50 |    7.00 | Свободно (7.0 ед.)


In [4]:
def solve_dual(effects, A_ub, b_ub):
    """Собирает и решает двойственную задачу для модели с верхними границами."""

    n_programs = len(effects)
    n_resources = len(b_ub)

    # Переменные двойственной задачи:
    # y_i >= 0 - теневые цены ресурсов (i = 1..n_resources)
    # z_j >= 0 - теневые цены ограничений x_j <= 1 (j = 1..n_programs)

    # Целевая функция двойственной задачи: min b^T y + sum(z_j)
    c_dual = np.concatenate([b_ub, np.ones(n_programs)])

    # Ограничения двойственной задачи:
    # A_ub^T y + z >= effects (для каждого j)
    # y_i >= 0, z_j >= 0

    # Матрица ограничений: [A_ub^T | I] * [y; z] >= effects
    A_dual = np.hstack([A_ub.T, np.eye(n_programs)])
    b_dual = effects

    # Границы двойственных переменных: все >= 0
    bounds_dual = [(0, None)] * (n_resources + n_programs)

    # Решаем двойственную задачу
    result_dual = linprog(
        c_dual,
        A_ub=-A_dual,  # Переходим к виду <=
        b_ub=-b_dual,
        bounds=bounds_dual,
        method='highs'
    )

    if not result_dual.success:
        raise RuntimeError(f"Dual solver failed: {result_dual.message}")

    return result_dual

# Решаем двойственную задачу
try:
    dual_result = solve_dual(effects, A_ub, b_ub)

    print("\n=== ДВОЙСТВЕННАЯ ЗАДАЧА ===\n")
    print(f"Статус: {dual_result.message}")
    print(f"Оптимальное значение двойственной задачи: {dual_result.fun:.2f}")

    # Разделяем переменные
    n_resources = len(b_ub)
    n_programs = len(effects)

    y_dual = dual_result.x[:n_resources]
    z_dual = dual_result.x[n_resources:]

    print("\nТеневые цены ресурсов (y_i):")
    for i, name in enumerate(resource_names):
        print(f"  {name}: {y_dual[i]:.4f}")

    print("\nТеневые цены ограничений x_j <= 1 (z_j):")
    for j, name in enumerate(program_names):
        print(f"  {name}: {z_dual[j]:.4f}")

    # Проверка сильной двойственности
    primal_value = -result.fun
    dual_value = dual_result.fun
    print(f"\n=== ПРОВЕРКА СИЛЬНОЙ ДВОЙСТВЕННОСТИ ===")
    print(f"Значение прямой задачи: {primal_value:.4f}")
    print(f"Значение двойственной задачи: {dual_value:.4f}")
    print(f"Разница: {abs(primal_value - dual_value):.10f}")
    print(f"Совпадение: {'ДА ✓' if np.allclose(primal_value, dual_value) else 'НЕТ ✗'}")

except Exception as e:
    print(f"Ошибка при решении двойственной задачи: {e}")
    dual_result = None


=== ДВОЙСТВЕННАЯ ЗАДАЧА ===

Статус: Optimization terminated successfully. (HiGHS Status 7: Optimal)
Оптимальное значение двойственной задачи: 259.73

Теневые цены ресурсов (y_i):
  Ресурс 1: 2.0909
  Ресурс 2: 0.0000
  Ресурс 3: 0.0000

Теневые цены ограничений x_j <= 1 (z_j):
  Программа 1: 0.8182
  Программа 2: 25.8182
  Программа 3: 32.3636
  Программа 4: 0.0000

=== ПРОВЕРКА СИЛЬНОЙ ДВОЙСТВЕННОСТИ ===
Значение прямой задачи: 259.7273
Значение двойственной задачи: 259.7273
Разница: 0.0000000000
Совпадение: ДА ✓


In [5]:
def rerun_with_resource_change(effects, A_ub, b_ub, bounds, resource_index, delta):
    """Пересчитывает модель после изменения одного ресурсного лимита."""

    # Копируем и изменяем лимит ресурса
    new_b_ub = b_ub.copy()
    new_b_ub[resource_index] += delta

    # Решаем задачу с новыми ограничениями
    c = -effects
    result_new = linprog(
        c,
        A_ub=A_ub,
        b_ub=new_b_ub,
        bounds=bounds,
        method='highs'
    )

    if not result_new.success:
        raise RuntimeError(f"New solver failed: {result_new.message}")

    return new_b_ub, result_new

In [6]:
print("СЦЕНАРИЙ 1: Увеличение бюджета на 5 единиц")

if result is not None and shadow_prices is not None:
    resource_idx = 0  # Бюджет
    delta = 5.0

    print(f"\nИсходный лимит бюджета: {b_ub[resource_idx]:.0f}")
    print(f"Изменение: +{delta:.0f}")
    print(f"Новый лимит: {b_ub[resource_idx] + delta:.0f}")

    try:
        new_b, new_result = rerun_with_resource_change(
            effects, A_ub, b_ub, bounds, resource_idx, delta
        )

        old_value = -result.fun
        new_value = -new_result.fun

        print(f"\nИсходный эффект: {old_value:.2f}")
        print(f"Новый эффект: {new_value:.2f}")
        print(f"Фактическое изменение: {new_value - old_value:.2f}")

        # Прогноз по теневой цене
        shadow_price = shadow_prices[resource_idx]
        predicted_change = shadow_price * delta
        print(f"\nПрогноз по теневой цене ({shadow_price:.4f} * {delta:.0f}): {predicted_change:.2f}")

        print(f"\nРазница между прогнозом и фактом: {abs(predicted_change - (new_value - old_value)):.4f}")
        print(f"Прогноз {'ТОЧНЫЙ ✓' if np.isclose(predicted_change, new_value - old_value, rtol=1e-4) else 'ПРИБЛИЖЕННЫЙ (локальная оценка)'}")

        print("\nНовый оптимальный план:")
        for i, name in enumerate(program_names):
            print(f"  {name}: {new_result.x[i]:.3f} ({new_result.x[i]*100:.1f}%)")

    except Exception as e:
        print(f"Ошибка в сценарии: {e}")

СЦЕНАРИЙ 1: Увеличение бюджета на 5 единиц

Исходный лимит бюджета: 96
Изменение: +5
Новый лимит: 101

Исходный эффект: 259.73
Новый эффект: 262.71
Фактическое изменение: 2.99

Прогноз по теневой цене (2.0909 * 5): 10.45

Разница между прогнозом и фактом: 7.4675
Прогноз ПРИБЛИЖЕННЫЙ (локальная оценка)

Новый оптимальный план:
  Программа 1: 1.000 (100.0%)
  Программа 2: 1.000 (100.0%)
  Программа 3: 1.000 (100.0%)
  Программа 4: 0.214 (21.4%)


In [7]:
print("СЦЕНАРИЙ 2: Увеличение трудозатрат на 3 единицы")

if result is not None and shadow_prices is not None:
    resource_idx = 1  # Трудозатраты
    delta = 3.0

    print(f"\nИсходный лимит трудозатрат: {b_ub[resource_idx]:.0f}")
    print(f"Изменение: +{delta:.0f}")
    print(f"Новый лимит: {b_ub[resource_idx] + delta:.0f}")

    try:
        new_b, new_result = rerun_with_resource_change(
            effects, A_ub, b_ub, bounds, resource_idx, delta
        )

        old_value = -result.fun
        new_value = -new_result.fun

        print(f"\nИсходный эффект: {old_value:.2f}")
        print(f"Новый эффект: {new_value:.2f}")
        print(f"Фактическое изменение: {new_value - old_value:.2f}")

        # Прогноз по теневой цене
        shadow_price = shadow_prices[resource_idx]
        predicted_change = shadow_price * delta
        print(f"\nПрогноз по теневой цене ({shadow_price:.4f} * {delta:.0f}): {predicted_change:.2f}")

        print(f"\nРазница между прогнозом и фактом: {abs(predicted_change - (new_value - old_value)):.4f}")

        if shadow_price > 0:
            print("Ресурс является дефицитным (binding)")
            print(f"Каждая дополнительная единица ресурса дает {shadow_price:.4f} ед. эффекта")
        else:
            print("Ресурс не является дефицитным (slack)")

        print("\nНовый оптимальный план:")
        for i, name in enumerate(program_names):
            print(f"  {name}: {new_result.x[i]:.3f} ({new_result.x[i]*100:.1f}%)")

    except Exception as e:
        print(f"Ошибка в сценарии: {e}")

СЦЕНАРИЙ 2: Увеличение трудозатрат на 3 единицы

Исходный лимит трудозатрат: 60
Изменение: +3
Новый лимит: 63

Исходный эффект: 259.73
Новый эффект: 259.73
Фактическое изменение: 0.00

Прогноз по теневой цене (0.0000 * 3): 0.00

Разница между прогнозом и фактом: 0.0000
Ресурс не является дефицитным (slack)

Новый оптимальный план:
  Программа 1: 1.000 (100.0%)
  Программа 2: 1.000 (100.0%)
  Программа 3: 1.000 (100.0%)
  Программа 4: 0.182 (18.2%)


In [8]:
print("СЦЕНАРИЙ 3: Увеличение эффекта Пунктов обогрева на 10%")

if result is not None:
    program_idx = 0  # Пункты обогрева
    new_effect = effects[program_idx] * 1.1

    print(f"\nИсходный эффект программы '{program_names[program_idx]}': {effects[program_idx]:.0f}")
    print(f"Новый эффект: {new_effect:.1f} (+10%)")

    try:
        # Изменяем вектор эффектов
        new_effects = effects.copy()
        new_effects[program_idx] = new_effect

        # Решаем задачу с новыми эффектами
        c_new = -new_effects
        result_new_c = linprog(
            c_new,
            A_ub=A_ub,
            b_ub=b_ub,
            bounds=bounds,
            method='highs'
        )

        if not result_new_c.success:
            raise RuntimeError(f"New solver failed: {result_new_c.message}")

        old_value = -result.fun
        new_value = -result_new_c.fun

        print(f"\nИсходный эффект: {old_value:.2f}")
        print(f"Новый эффект: {new_value:.2f}")
        print(f"Изменение: {new_value - old_value:.2f}")

        print("\nНовый оптимальный план:")
        for i, name in enumerate(program_names):
            print(f"  {name}: {result_new_c.x[i]:.3f} ({result_new_c.x[i]*100:.1f}%)")

    except Exception as e:
        print(f"Ошибка в сценарии: {e}")

СЦЕНАРИЙ 3: Увеличение эффекта Пунктов обогрева на 10%

Исходный эффект программы 'Программа 1': 97
Новый эффект: 106.7 (+10%)

Исходный эффект: 259.73
Новый эффект: 269.43
Изменение: 9.70

Новый оптимальный план:
  Программа 1: 1.000 (100.0%)
  Программа 2: 1.000 (100.0%)
  Программа 3: 1.000 (100.0%)
  Программа 4: 0.182 (18.2%)


In [9]:
print("ИТОГОВЫЙ АНАЛИЗ И ИНТЕРПРЕТАЦИЯ")
print("СОЦИАЛЬНАЯ ЗАЩИТА В ЗИМНИЙ ПЕРИОД")

if result is not None and shadow_prices is not None and dual_result is not None:

    print("\n1. ПРЯМАЯ ЗАДАЧА:")
    print(f"   Максимальный эффект социальной защиты: {-result.fun:.2f}")
    print("\n   Оптимальный план:")
    for i, name in enumerate(program_names):
        print(f"     • {name}: {result.x[i]:.3f} (использовано {result.x[i]*100:.1f}% от максимума)")

    print("\n2. АКТИВНЫЕ ОГРАНИЧЕНИЯ (ЗИМНИЙ ПЕРИОД):")
    for i, name in enumerate(resource_names):
        used = (A_ub[i] * result.x).sum()
        is_binding = np.isclose(slack[i], 0.0, atol=1e-6)
        status = "АКТИВНО (binding)" if is_binding else f"Свободно (остаток {slack[i]:.1f})"
        print(f"     • {name}: использовано {used:.1f}/{b_ub[i]:.0f} - {status}")

    print("\n3. ТЕНЕВЫЕ ЦЕНЫ РЕСУРСОВ (y_i):")
    n_resources = len(b_ub)
    y_dual = dual_result.x[:n_resources]
    for i, name in enumerate(resource_names):
        print(f"     • {name}: {y_dual[i]:.4f}")
        print(f"       Единица измерения: ед. эффекта / ед. {name}")

    print("\n4. ТЕНЕВЫЕ ЦЕНЫ ВЕРХНИХ ГРАНИЦ (z_j):")
    n_programs = len(effects)
    z_dual = dual_result.x[n_resources:]
    for j, name in enumerate(program_names):
        if z_dual[j] > 0.001:
            print(f"     • {name}: {z_dual[j]:.4f}")
            print(f"       Единица измерения: ед. эффекта / ед. масштаба")

    print("\n5. ГЛАВНОЕ УЗКОЕ МЕСТО ЗИМОЙ:")
    # Находим самый дефицитный ресурс
    max_shadow_idx = np.argmax(y_dual)
    print(f"   • Самый дефицитный ресурс: {resource_names[max_shadow_idx]}")
    print(f"     Теневая цена: {y_dual[max_shadow_idx]:.4f}")
    print(f"     Это означает, что увеличение {resource_names[max_shadow_idx]} на 1 единицу")
    print(f"     принесет дополнительный эффект {y_dual[max_shadow_idx]:.2f} единиц.")

    print("\n6. ИНТЕРПРЕТАЦИЯ ДЛЯ ЗИМНЕГО ПЕРИОДА:")
    print("   В условиях зимы наиболее критичными ресурсами являются:")
    for i, name in enumerate(resource_names):
        if y_dual[i] > 0.01:
            print(f"   • {name}: теневая цена {y_dual[i]:.4f} - дефицитный ресурс")
        else:
            print(f"   • {name}: теневая цена {y_dual[i]:.4f} - есть запас")

    print("\n7. УПРАВЛЕНЧЕСКИЕ РЕКОМЕНДАЦИИ:")
    if y_dual[max_shadow_idx] > 0:
        print(f"   • Приоритет: увеличить {resource_names[max_shadow_idx]}")
        print(f"   • Ожидаемый эффект: {y_dual[max_shadow_idx]:.2f} ед. на каждую доп. единицу ресурса")

    # Анализируем программы с z_j > 0
    active_programs = [j for j, z in enumerate(z_dual) if z > 0.001]
    if active_programs:
        print("   • Рекомендуется рассмотреть расширение программ:")
        for j in active_programs:
            print(f"     - {program_names[j]} (доп. эффект от расширения: {z_dual[j]:.4f})")

    print("\n8. ЛОКАЛЬНОСТЬ ПРОГНОЗА:")
    print("   Прогноз по теневой цене является ЛОКАЛЬНЫМ.")
    print("   Для больших изменений ресурсов прогноз может отличаться")
    print("   от фактического пересчета.")
    print("   Рекомендуется проверять значительные изменения повторным решением.")

    print("\n9. ВЫВОД ПО ЗИМНЕМУ ПЕРИОДУ:")
    print("   Для эффективной социальной защиты в зимний период")
    print("   необходимо сконцентрироваться на обеспечении:")
    if y_dual[max_shadow_idx] > 0:
        print(f"   • {resource_names[max_shadow_idx]} как ключевого ресурса")
    else:
        print("   • Все ресурсы имеют запас, структура оптимальна")
    print("   • Приоритетные программы:")
    # Сортируем программы по эффективности
    program_effectiveness = []
    for j in range(len(program_names)):
        if result.x[j] > 0.001:
            efficiency = effects[j] / A_ub[:, j].sum()
            program_effectiveness.append((program_names[j], result.x[j], efficiency))
    program_effectiveness.sort(key=lambda x: x[2], reverse=True)
    for name, scale, eff in program_effectiveness[:3]:
        print(f"     - {name}: масштаб {scale*100:.0f}%, эффективность {eff:.2f}")

ИТОГОВЫЙ АНАЛИЗ И ИНТЕРПРЕТАЦИЯ
СОЦИАЛЬНАЯ ЗАЩИТА В ЗИМНИЙ ПЕРИОД

1. ПРЯМАЯ ЗАДАЧА:
   Максимальный эффект социальной защиты: 259.73

   Оптимальный план:
     • Программа 1: 1.000 (использовано 100.0% от максимума)
     • Программа 2: 1.000 (использовано 100.0% от максимума)
     • Программа 3: 1.000 (использовано 100.0% от максимума)
     • Программа 4: 0.182 (использовано 18.2% от максимума)

2. АКТИВНЫЕ ОГРАНИЧЕНИЯ (ЗИМНИЙ ПЕРИОД):
     • Ресурс 1: использовано 96.0/96 - АКТИВНО (binding)
     • Ресурс 2: использовано 59.1/60 - Свободно (остаток 0.9)
     • Ресурс 3: использовано 43.0/50 - Свободно (остаток 7.0)

3. ТЕНЕВЫЕ ЦЕНЫ РЕСУРСОВ (y_i):
     • Ресурс 1: 2.0909
       Единица измерения: ед. эффекта / ед. Ресурс 1
     • Ресурс 2: 0.0000
       Единица измерения: ед. эффекта / ед. Ресурс 2
     • Ресурс 3: 0.0000
       Единица измерения: ед. эффекта / ед. Ресурс 3

4. ТЕНЕВЫЕ ЦЕНЫ ВЕРХНИХ ГРАНИЦ (z_j):
     • Программа 1: 0.8182
       Единица измерения: ед. эффекта / ед. м

## 6. Что должно быть в отчёте

1. Прямая постановка задачи.
2. Объяснение двойственной задачи как зеркала прямой.
3. Оптимальный план по программам.
4. Таблица активных ограничений и запасов.
5. Таблица теневых цен ресурсов.
6. Единица измерения каждой теневой цены.
7. Проверка сильной двойственности.
8. Объяснение нулевых shadow prices, если они есть.
9. Разделение смысла `y_i` и `z_j`.
10. Минимум два сценария по `b` и один по `c`.
11. Отметка, что прогноз по shadow price локален.

## 7. Контрольный чек-лист

- [ ] Я показал, какие ограничения стали binding.
- [ ] Я объяснил dual как зеркальный взгляд на primal.
- [ ] Я указал единицу измерения для каждой теневой цены.
- [ ] Я объяснил, что нулевая теневая цена не делает ресурс бесполезным вообще.
- [ ] Я отделил ресурсные переменные `y_i` от переменных верхних границ `z_j`.
- [ ] Я осмысленно интерпретировал shadow prices.
- [ ] Я сравнил прогноз и фактический пересчёт.
- [ ] Я сформулировал управленческий вывод по самому дефицитному ресурсу.